## Marco del análisis

- **Qué estimamos:** la elasticidad precio-demanda propia de Pepsi 2L.
- **Target y grano:** ln(unidades) a nivel tienda·semana (panel, sin agregar).
- **Pregunta de pricing:** optimizar ingreso (el margen queda como posible extensión).
- **Confusores a controlar:** promoción, festivos, estacionalidad y tienda.
- **Naturaleza de los datos:** observacionales; la elasticidad es una asociación condicionada a los controles, válida dentro del rango de precios histórico. Amenaza principal: endogeneidad del precio.
- **Enfoque:** inferencia (calidad del coeficiente), no predicción. Partimos de la regresión ingenua y añadimos controles paso a paso.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


import statsmodels.formula.api as smf

### Carga de datos

In [2]:
pepsi = pd.read_parquet('../data/processed/pepsi_2l.parquet')
print(pepsi.shape)
print(pepsi.columns.tolist())

(31751, 15)
['STORE', 'UPC', 'WEEK', 'MOVE', 'QTY', 'PRICE', 'SALE', 'PROFIT', 'DESCRIP', 'SIZE', 'COM_CODE', 'start', 'end', 'special', 'UNIT_PRICE']


In [3]:
pepsi['UNIT_PRICE'] = pepsi['PRICE'] / pepsi['QTY'] # calculamos precio unitario
# aplicamos logaritmo a unidades vendidas y precio unitario para calcular elasticidad
pepsi['ln_q'] = np.log(pepsi['MOVE'])
pepsi['ln_p'] = np.log(pepsi['UNIT_PRICE'])

In [4]:
# Transformamos en variable dummy las promociones y festivos
pepsi['PROMO'] = pepsi['SALE'].notna().astype(int)
pepsi['FESTIVO'] = pepsi['special'].str.strip().notna().astype(int)

# Transformamos variable tiendas en categórica
pepsi['STORE'] = pepsi['STORE'].astype('category')


Para trabajar con las variables `SALE` y `special`, las transformamos en variables binarias, ya que no nos interesa en este momento que es cada cosa, simplemente aislar su efecto de la forma más sencilla posible. En `SALE` se registra el tipo de descuento aplicado al producto y en `special` el tipo de festividad que hay en esa semana.

Finalmente, como las tiendas se registran como número, las transformamos en variable categórica para que llegado el momento de aplicar la regresión, no interprete que una tienda tiene más peso que otra por tener diferente número.

Con el dataset estructurado, pasamos a la fase de regresión, en donde se van a generar varias regresiones añadiendo cada vez más complejidad con el efecto que provocan las promociones y días festivos. De esta forma se puede ir evaluando como cambia la elasticidad.

## Regresión

In [44]:
# inicializamos lista para guardar resultados
resultados = []

# Regresión 1 - Ingenua. Relación precio-cantidad sin ningún control.

m0 = smf.ols('ln_q ~ ln_p', data=pepsi).fit() # instanciamos y ajustamos modelo
resultados.append(('1. Ingenua', m0.params['ln_p']))
print(f"Elasticidad ingenua: {m0.params['ln_p']:.4f}")

Elasticidad ingenua: -4.0217


In [45]:
# Regresión con promoción incluida
m1 = smf.ols('ln_q ~ ln_p + PROMO', data=pepsi).fit()
resultados.append(('2. Promoción', m1.params['ln_p']))
print(f"Elasticidad (con promo): {m1.params['ln_p']:.4f}")
print(f"Efecto promoción: {m1.params['PROMO']:.4f} (p = {m1.pvalues['PROMO']:.3f})")

Elasticidad (con promo): -4.0496
Efecto promoción: -0.0176 (p = 0.077)


Los resultados de los coeficientes en esta segunda regresión son bastante contradictorios con lo que se espera, que es una bajada de la elasticidad, contrariamente de lo que pasa, que es un aumento de la misma. Este resultado junto a la no significatividad del efecto promoción en la regresión contradice los hallazgos del EDA, donde las semanas en promoción vendían mucho más que aquellas que registraban un precio regular. 

In [46]:
# Cuánto se solapan precio y promoción?
pepsi.groupby("PROMO")["ln_p"].describe()[["mean", "min", "max"]]

,mean,min,max
PROMO,,,
0,0.407507,-0.235722,0.636577
1,0.155039,-0.385662,0.636577


Como ya hemos codificado el efecto de promoción en binario, vamos a examinar esta variable más de cerca, ya que la variable original en la documentación de los propios datos nos indica que puede quedar algún error en las promociones. Si calculamos el precio medio, se puede observar como aquellos precios con promocion (`1`) tienen un precio mucho más bajo que los precios regulares. Observando el rango de precio, que ambas categorías coincidan exactamente en el valor máximo, nos indica lo que ya se sospechaba e indicaba la documentación, posibles errores en el registro de promociones.

In [47]:
pepsi.groupby("PROMO")["MOVE"].mean()

PROMO
0    161.755329
1    508.268573
Name: MOVE, dtype: float64

In [48]:
# Semanas marcadas como promo pero con precio alto: ¿tienen sentido?
pepsi[pepsi["PROMO"] == 1].sort_values("ln_p", ascending=False)[
    ["STORE", "start", "UNIT_PRICE", "MOVE", "SALE", "PROMO"]
].head(10)

,STORE,start,UNIT_PRICE,MOVE,SALE,PROMO
210219,14,1991-07-11,1.89,44,S,1
224222,130,1991-07-11,1.89,157,S,1
213094,53,1991-07-11,1.89,23,S,1
218704,95,1991-07-11,1.89,107,S,1
214435,68,1991-07-11,1.89,87,S,1
214820,71,1991-07-11,1.89,62,S,1
215015,72,1991-07-11,1.89,40,S,1
213876,62,1991-07-11,1.89,30,S,1
218323,93,1991-07-11,1.89,51,S,1
225045,137,1991-07-11,1.89,48,S,1


Inspeccionando las semanas marcadas como promción podemos ver la causa, hay registros marcados como promción pero que cuentan con el precio más elevado y el número de ventas es bajo.

Para tratar de corregir esta situación vamos a reconstruir la señal de promoción a partir del propio precio. Para reconstruir el precio lo que se hace es definir un **precio de referencia** por tienda, el precio regular del producto en cada tienda, y se mide la **profundiad de descuento** como la caída del precio observado respecto a la referencia. 

Para construir el precio de referencia vamos a emplear un cuantil alto (percentil 90), es que donde se encuentran los precios altos, para una ventana móvil de 13 semanas, haciendolo coincidir con el trimestre, calculado por tienda. La lógica de esta decisión es que como las promociones bajan el precio y son frecuentes en la serie, el precio regular siempre vive en la parte alta de la distribución local. Por tanto un cuantil alto captura mejor que la mediana, que se vería contaminada con la cantidad de semanas que tiene el precio descontado. La ventana móvil permite que la referencia de precio se adapte al cambio que se vaya ocasionando a lo largo de la serie, de esta forma evitamos establecer un precio fijo para los 6 años.

## Feature Engineering

In [49]:
# Nos aseguramos de que todas las tiendas estén bien agrupadas por fecha.
pepsi_sorted = pepsi.sort_values(['STORE','start']).reset_index(drop=True)

# Parámetros de ventana móvil
VENTANA = 13 # semanas coincidente con trimestre
CUANTIL = 0.90 # parte más alta de la distribución

# creamos nueva variable con precio referencia
pepsi_sorted['PRECIO_REF'] = (
    pepsi_sorted.groupby('STORE', observed=True)['UNIT_PRICE'] # aplicamos el cálculo separado por tienda
    .transform(lambda x: x.rolling(VENTANA, min_periods=5, center=True) # Con center indicamos que mire tanto atrás como adelante
               .quantile(CUANTIL))
)

# Comprobamos que los precios se han construido de forma correcta y se encuentran en la parte alta de la distrbución
display(pepsi_sorted['PRECIO_REF'].describe())
display(pepsi_sorted[['UNIT_PRICE', 'PRECIO_REF']].sample(10))
print(f'Precios de referencia vacíos: {pepsi_sorted["PRECIO_REF"].isna().sum()}')


count    31751.000000
mean         1.580196
std          0.116964
min          1.180000
25%          1.490000
50%          1.590000
75%          1.690000
max          1.890000
Name: PRECIO_REF, dtype: float64

,UNIT_PRICE,PRECIO_REF
22771,0.89,1.490
16960,1.40,1.490
10645,1.49,1.490
8900,1.79,1.790
20281,1.69,1.690
24302,1.49,1.490
13715,1.39,1.390
23622,1.19,1.588
11862,0.99,1.590
30662,0.99,1.490


Precios de referencia vacíos: 0


Una vez se ha construido el **precio de referencia** y observando los resultados se determina que se ha reconstruido la variable de forma correcta. Los cuartiles de la variable se encuentran en la parte alta de la distrbución del precio original. Si observamos una pequeña muestra, se puede ver como todos los precios de referencia son mayores o iguales al precio original, justo lo que se buscaba. Finalmente hacemos una comprobación rápida para asegurarnos de que no quedaron espacios en blanco dentro del prefio de referencia.

El siguiente paso es calcular el descuento aplicado, es decir, la diferencia entre el precio de referencia y el precio en logaritmo. Como la diferencia de logaritmos es aproximadamente un cambio porcentual, la variable que se va a crear se puede leer directamente como **qué porcentaje por debajo de su precio regular está el precio esta semana**.

Con esta nueva variable será posible arreglar el problema que presentaba la segunda regresión, en donde no se diferenciaba precio de promoción, dando al modelo prácticamente la misma información.

In [50]:
# creamos descuento
pepsi_sorted['DESCUENTO'] = np.log(pepsi_sorted['PRECIO_REF']) - pepsi_sorted['ln_p']

# Comprobamos resutlados
display(pepsi_sorted['DESCUENTO'].describe())
print(f"Cantidad de descuentos negativos: {(pepsi_sorted['DESCUENTO'] < -0.01).sum()}")

count    31751.000000
mean         0.159883
std          0.198788
min         -0.153165
25%          0.000000
50%          0.045024
75%          0.312598
max          0.898486
Name: DESCUENTO, dtype: float64

Cantidad de descuentos negativos: 139


Las estadísticas del descuento nos indican que hay precios negativos, con un mínimo del 15% (-0.15) por encima de su propio precio regular. Si observamos el número total de descuentos negativos obtenemos un resultado de 139, algo que conceptualmente no debería ocurrir. 

Lo más probable es que se hubiese producido un cambio de precio y el precio de referencia creado aún no hubiese capturado el cambio. Antes de seguir vamos a comprobar que el problema está en el cambio de escalón.

In [51]:
pepsi_sorted[pepsi_sorted["DESCUENTO"] < -0.01][["STORE", "start", "UNIT_PRICE", "PRECIO_REF", "DESCUENTO"]].head(10)

,STORE,start,UNIT_PRICE,PRECIO_REF,DESCUENTO
189,2,1993-06-17,1.69,1.530,-0.099461
344,2,1996-08-08,1.69,1.670,-0.011905
836,8,1991-11-07,1.49,1.390,-0.069472
917,8,1993-06-10,1.69,1.650,-0.023953
918,8,1993-06-17,1.69,1.650,-0.023953
1073,8,1996-08-08,1.69,1.548,-0.087765
1080,8,1996-09-26,1.59,1.536,-0.034552
1300,9,1993-06-10,1.68,1.642,-0.022879
1686,12,1993-06-17,1.59,1.506,-0.054277
1741,12,1994-07-14,1.59,1.570,-0.012658


Comprobando los descuentos que salen en negativo se puede ver como el error se produce por un cambio de precio en `UNITE PRICE` que el precio de referencia no llega a capturar a tiempo, nada grave de lo que preocuparse. Para evitar que provoquen errores en la regresión vamos a llevar estos valores negativos a 0, indicando así que para esas referencias no hubo descuento en esa semana.

In [52]:
pepsi_sorted['DESCUENTO'] = pepsi_sorted['DESCUENTO'].clip(lower=0) # llevamos descuentos negativos a 0
print(f"Cantidad de descuentos negativos: {(pepsi_sorted['DESCUENTO'] < -0.01).sum()}")

Cantidad de descuentos negativos: 0


Ahora que el se ha reconstruido tanto el precio como el descuento ya es posible volver a trabajar en la regresión para calcular la elasticidad y medir el efecto del descuento.

## Regresión con feature engineering

In [53]:
m1b = smf.ols('ln_q ~ ln_p + DESCUENTO', data=pepsi_sorted).fit()
resultados.append(('2. Descuento', m1b.params['ln_p']))
print(f"Elasticidad:        {m1b.params['ln_p']:.4f}")
print(f"Efecto descuento:   {m1b.params['DESCUENTO']:.4f} (p = {m1b.pvalues['DESCUENTO']:.3f})")
print(m1b.summary())

Elasticidad:        -4.2617
Efecto descuento:   -0.2584 (p = 0.000)
                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.577
Model:                            OLS   Adj. R-squared:                  0.577
Method:                 Least Squares   F-statistic:                 2.165e+04
Date:               vi., 11 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:38:53   Log-Likelihood:                -33042.
No. Observations:               31751   AIC:                         6.609e+04
Df Residuals:                   31748   BIC:                         6.612e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------

Al realizar la regresión con el descuento sobre el precio original, volvemos a tener el mismo problema que en el primer planteamiento, tanto el precio real como el descuento se solapan, dando lugar a que ambas variables comparten la misma información. Si observamos los coeficientes, el DESCUENTO sale con signo negativo. Más que interpretarlo (descontar no puede reducir las ventas, sabemos por el EDA que las dispara), lo leemos como la señal de que esta especificación está mal planteada: al solaparse precio y descuento, el coeficiente del descuento no tiene una interpretación fiable.

Antes de sacar conclusiones precipitadas vamos a plantear una fórmula para el modelo diferente, en donde aplicaremos el logaritmo del `PRECIO_REF`, que es la elasticidad respecto al precio regular, es decir, cómo responder la demanda cuando cambia el precio de lista, no el de oferta.

In [54]:
m1c = smf.ols('ln_q ~ np.log(PRECIO_REF) + DESCUENTO', data=pepsi_sorted).fit()
resultados.append(('2. PRECIO_REF', m1c.params['np.log(PRECIO_REF)']))
print(m1c.summary())
print(f"Elasticidad: {m1c.params['np.log(PRECIO_REF)']:.4f}")
print(f"Efecto descuento: {m1c.params['DESCUENTO']:.4f} (p = {m1b.pvalues['DESCUENTO']:.3f})")

                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.577
Model:                            OLS   Adj. R-squared:                  0.577
Method:                 Least Squares   F-statistic:                 2.163e+04
Date:               vi., 11 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:38:59   Log-Likelihood:                -33049.
No. Observations:               31751   AIC:                         6.610e+04
Df Residuals:                   31748   BIC:                         6.613e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              6.4174      0

Cambiando el modelo, se puede ver como el `DESCUENTO` finalmente cobra sentido, obteniendo un resultado muy significativo en consonancia con los hallazgos obtenidos en el EDA. Ahora se aprecia como un precio con descuento dispara las ventas del producto.

En cuanto al $R^2$ vemos que este sigue siendo idéntico. La mejora no está en que el modelo sea capaz de explicar más la varianza, sino en repartir de forma correcta los efectos entre precio y descuento. De todas formas, este no es un modelo definitivo ya que aún quedan variables pendientes de añadir.

A continuación se añade la variable `FESTIVO` como predictor.

In [59]:
m2 = smf.ols('ln_q ~ np.log(PRECIO_REF) + DESCUENTO + FESTIVO', data=pepsi_sorted).fit()
resultados.append(('3. FESTIVO', m2.params['np.log(PRECIO_REF)']))
print(m2.summary())
print(f"Elasticidad (precio regular): {m2.params['np.log(PRECIO_REF)']:.4f}")
print(f"Efecto descuento: {m2.params['DESCUENTO']:.4f}")
print(f"Efecto festivo: {m2.params['FESTIVO']:.4f} (p = {m2.pvalues['FESTIVO']:.3f})")

                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.577
Model:                            OLS   Adj. R-squared:                  0.577
Method:                 Least Squares   F-statistic:                 1.443e+04
Date:               vi., 11 sep. 2026   Prob (F-statistic):               0.00
Time:                        11:47:19   Log-Likelihood:                -33043.
No. Observations:               31751   AIC:                         6.609e+04
Df Residuals:                   31747   BIC:                         6.613e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              6.4235      0

El efecto festivo, a pesar de ser significativo, tiene un impacto muy poco relevante, se puede interpretar como que en una semana con festivo hay un -3% menos de ventas. En este caso, como el efecto no importa mucho, es una variable candidata a ser eliminada del modelo final, de esta forma se simplifica el modelo. 

Antes de eliminar variables, queda pendiente diferenciar por tienda, que es el siguiente paso.

In [62]:
m3 = smf.ols('ln_q ~ np.log(PRECIO_REF) + DESCUENTO + FESTIVO + STORE', data=pepsi_sorted).fit()

print(m3.summary())
print(f"Elasticidad (precio regular): {m3.params['np.log(PRECIO_REF)']:.4f}")
print(f"Efecto descuento: {m3.params['DESCUENTO']:.4f}")
print(f"Efecto festivo: {m3.params['FESTIVO']:.4f}")

                            OLS Regression Results                            
Dep. Variable:                   ln_q   R-squared:                       0.716
Model:                            OLS   Adj. R-squared:                  0.715
Method:                 Least Squares   F-statistic:                     839.0
Date:               vi., 11 sep. 2026   Prob (F-statistic):               0.00
Time:                        12:03:42   Log-Likelihood:                -26731.
No. Observations:               31751   AIC:                         5.365e+04
Df Residuals:                   31655   BIC:                         5.446e+04
Df Model:                          95                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              5.8745      0